<a href="https://colab.research.google.com/github/Protasovkr/quest2_v1/blob/main/04_%D0%9E%D1%86%D0%B5%D0%BD%D0%BA%D0%B0_%D0%BA%D0%B0%D1%87%D0%B5%D1%81%D1%82%D0%B2%D0%B0_%D0%BA%D0%BB%D0%B0%D1%81%D1%81%D0%B8%D1%84%D0%B8%D0%BA%D0%B0%D1%86%D0%B8%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report

# =====================================================================
# 1. ЗАГРУЗКА И АНАЛИЗ ДАННЫХ
# =====================================================================

# Предполагается, что файлы train.csv и test.csv загружены в файлы Colab
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Размерность тренировочных данных: {train.shape}")
print(f"Размерность тестовых данных: {test.shape}")

# Быстрый просмотр первых строк
print(train.head())

# Проверка на пропуски (как на [00:41:12])
print("Проверка на пропуски:")
print(train.isna().sum())

# Информация о типах данных
print(train.info())

# Проверка сбалансированности целевой переменной 'is_canceled' ([00:42:27])
print(train['is_canceled'].value_counts())


# =====================================================================
# 2. ПРЕДОБРАБОТКА ДАННЫХ (ПЕРВЫЙ ВАРИАНТ - ХОРОШИЙ)
# =====================================================================

# Удаление признаков по "здравому смыслу" ([00:51:58])
# Лектор удаляет дату бронирования и страну
drop_cols = ['reservation_status_date', 'country']
train = train.drop(columns=drop_cols, errors='ignore')
test = test.drop(columns=drop_cols, errors='ignore')

# Разделение признаков и целевой переменной
X_train = train.drop(columns=['is_canceled'])
y_train = train['is_canceled']

# Для теста целевой переменной может не быть на платформе, берем все признаки
X_test = test.copy()

# --- Кодирование признаков через LabelEncoder ---
# Лектор показывает пример с кодированием 'arrival_date_month' ([00:55:19])
le = LabelEncoder()
X_train['arrival_date_month'] = le.fit_transform(X_train['arrival_date_month'])
# Важное правило из видео [01:17:52]: для теста используем только transform!
X_test['arrival_date_month'] = le.transform(X_test['arrival_date_month'])

# Применяем LabelEncoder к остальным текстовым колонкам (кроме тех, что пойдут в dummies)
le_cols = ['meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']
for col in le_cols:
    if col in X_train.columns:
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))

# --- Кодирование через dummy-переменные (One-Hot) ---
# На [00:58:20] лектор переводит категориальные переменные в бинарные столбцы
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

# Выравниваем столбцы трейна и теста (на случай расхождений категорий, как на [01:01:14])
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)


# =====================================================================
# 3. ОБУЧЕНИЕ МОДЕЛИ И ОЦЕНКА (ВАРИАНТ 1)
# =====================================================================

# Создаем и обучаем логистическую регрессию ([00:59:21])
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Предсказание на тренировочной выборке для построения матрицы ошибок
y_pred_train = model.predict(X_train)

# Вывод матрицы ошибок (Confusion Matrix) ([01:02:06])
print("Матрица ошибок (Вариант 1):")
print(confusion_matrix(y_train, y_pred_train))

# Метрики качества (Precision, Recall, F1)
print("\nОтчет о классификации (Вариант 1):")
print(classification_report(y_train, y_pred_train))

# Предсказание для теста (для отправки решения)
test_predictions = model.predict(X_test)
submission = pd.DataFrame({'is_canceled': test_predictions})
submission.to_csv('my_submission.csv', index=False)
print("\nФайл 'my_submission.csv' успешно сохранен. Ожидаемый скор ~ 0.748")


# =====================================================================
# 4. ЭКСПЕРИМЕНТ: "КАК ВСЁ ПЛОХО БЕЗ КАТЕГОРИЙ" (ВТОРОЙ ВАРИАНТ)
# =====================================================================
# На [01:03:13] лектор проводит эксперимент и полностью удаляет категориальные столбцы
print("\n--- Запуск второго эксперимента (удаление категорий) ---")

# Заново берем чистые числовые данные из исходных датасетов
# Выбираем только столбцы с типами int64 и float64
X_train_bad = train.drop(columns=['is_canceled']).select_dtypes(include=['int64', 'float64'])
X_test_bad = test.select_dtypes(include=['int64', 'float64'])

# Обучаем модель заново
model_bad = LogisticRegression(max_iter=1000)
model_bad.fit(X_train_bad, y_train)

y_pred_bad = model_bad.predict(X_train_bad)

print("Матрица ошибок (Вариант 2 - Без категорий):")
print(confusion_matrix(y_train, y_pred_bad))

print("\nОтчет о классификации (Вариант 2):")
print(classification_report(y_train, y_pred_bad))

# Скор падает примерно до 0.47, как было продемонстрировано на платформе ([01:05:00])